In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Online Retail Data Set.csv', encoding='ISO-8859-1')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850.0,United Kingdom


In [3]:
df.dropna(subset=['CustomerID'], inplace=True)

In [4]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

ValueError: time data "13-12-2010 09:02" doesn't match format "%m-%d-%Y %H:%M", at position 927. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [5]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%d-%m-%Y %H:%M')

In [6]:
df = df[df['Quantity'] > 0]

In [7]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 397924 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    397924 non-null  object        
 1   StockCode    397924 non-null  object        
 2   Description  397924 non-null  object        
 3   Quantity     397924 non-null  int64         
 4   InvoiceDate  397924 non-null  datetime64[ns]
 5   UnitPrice    397924 non-null  float64       
 6   CustomerID   397924 non-null  float64       
 7   Country      397924 non-null  object        
 8   TotalPrice   397924 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 30.4+ MB


In [9]:
df.drop_duplicates(inplace=True)

In [10]:
print(f"Final Row Count: {len(df)}")
df.head()

Final Row Count: 392732


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [11]:
df.to_csv("cleaned_transactions.csv", index=False)


NameError: name 'rfm' is not defined

In [12]:
df.to_csv("cleaned_transactions.csv", index=False)

In [13]:
import datetime as dt

snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

In [14]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})

In [15]:
rfm.columns = ['Recency', 'Frequency', 'Monetary']

In [16]:
rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


In [17]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

In [18]:
rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

In [19]:
def segment_customer(row):
    if row['RFM_Score'] == '444':
        return 'Best Customers'
    elif row['F_score'] == 4:
        return 'Loyal Customers'
    elif row['R_score'] == 4:
        return 'Recent Customers'
    else:
        return 'At Risk'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

In [21]:
rfm = rfm.reset_index()

In [22]:
rfm.to_csv("customer_rfm.csv", index=False)

In [23]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Select features
X = rfm[['Recency', 'Frequency', 'Monetary']]

# Scale data (VERY IMPORTANT)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply KMeans
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(X_scaled)

# View result
rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score,Segment,Cluster
0,12346.0,326,1,77183.60,1,1,4,114,At Risk,0
1,12347.0,2,7,4310.00,4,4,4,444,Best Customers,3
2,12348.0,75,4,1797.24,2,3,4,234,At Risk,3
3,12349.0,19,1,1757.55,3,1,4,314,At Risk,3
4,12350.0,310,1,334.40,1,1,2,112,At Risk,1


In [24]:
rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean()

,Recency,Frequency,Monetary
Cluster,,,
0,15.672986,22.047393,12435.086682
1,248.564030,1.551789,476.330547
2,7.384615,82.692308,127187.959231
3,43.910580,3.655748,1344.284013


In [25]:
def label_cluster(row):
    if row['Cluster'] == 2:
        return "VIP Customers"
    elif row['Cluster'] == 0:
        return "Loyal Customers"
    elif row['Cluster'] == 3:
        return "Occasional Customers"
    else:
        return "At Risk Customers"

rfm['Cluster_Label'] = rfm.apply(label_cluster, axis=1)

In [26]:
rfm.to_csv("customer_rfm_final.csv", index=False)